<a href="https://colab.research.google.com/github/hijyun/Overwatch-League-Prediction/blob/main/Prediction/mapdata_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ?? ??? ??

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import collections
import pandas as pd
from sklearn import datasets

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier # ?? ??
from sklearn.neighbors import KNeighborsClassifier # KNN
from sklearn.linear_model import LogisticRegression # ???? ??

from sklearn.ensemble import VotingClassifier,RandomForestClassifier,AdaBoostClassifier # ??? 

from sklearn.metrics import confusion_matrix, classification_report # ?????
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score 
from sklearn.metrics import roc_curve, auc, roc_auc_score  # ROC?? ???

from sklearn.model_selection import learning_curve, validation_curve # ????, ????
from sklearn.model_selection import GridSearchCV, cross_val_score  # ??????? ??, ?????

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# ??? ??

In [ ]:
df1 = pd.read_csv('https://raw.githubusercontent.com/hijyun/Overwatch-League-Prediction/main/preprocessing/e-sprot_data.csv')

In [ ]:
X = df1.drop(['target'],axis=1)
y = df1['target']

In [ ]:
df1.shape

In [ ]:
df1.head()

In [ ]:
df1.info()

# ??? ? ??? ??

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder() # ??? ??? 0?? ???? ??? ??? ??
y = le.fit_transform(y)


In [ ]:
y

## train data , test data split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=1234, stratify=y)

# ??

In [ ]:
logistic = LogisticRegression(solver='liblinear',penalty='l2',C=0.1,random_state=1)
tree = DecisionTreeClassifier(max_depth=5,criterion='entropy',random_state=1)
knn = KNeighborsClassifier(n_neighbors=2,p=2,metric='minkowski')

voting_estimators = [('logistic', logistic), ('tree', tree), ('knn', knn)]

voting = VotingClassifier(estimators = voting_estimators,voting='soft')

forest = RandomForestClassifier(criterion='entropy',n_estimators=700,random_state=1)


adaboost = AdaBoostClassifier(base_estimator=tree, 
                                n_estimators=300,
                                learning_rate = 0.01, 
                                random_state=42)
lgb = LGBMClassifier(random_state=42)
xgb = XGBClassifier(tree_method='hist',random_state=41)

clf_labels = ['Knn','Decision tree',  'Majority voting','Random forest','Ada boost','XGB','LIGHTGBM']

all_clf = [knn, tree, voting, forest,adaboost,xgb,lgb]

### AUC ??

In [ ]:
for clf, label in zip(all_clf, clf_labels):
  scores = cross_val_score(estimator=clf,X=X_train,y=y_train,cv=10,scoring='roc_auc')
  print("ROC AUC: %0.3f (+/- %0.3f) [%s]"
  % (scores.mean(), scores.std(), label))

### ROC ??

In [ ]:
colors = ['black', 'orange', 'blue', 'green','red','gray','skyblue']
linestyles = [':', '--', '-.', '-','solid','dashdot','dotted']
for clf, label, clr, ls \
        in zip(all_clf, clf_labels, colors, linestyles):

  clf.fit(X_train, y_train)
  y_pred = clf.predict_proba(X_test)[:, 1]
  fpr, tpr, thresholds = roc_curve(y_true=y_test,
                                  y_score=y_pred)
  roc_auc = auc(x=fpr, y=tpr) # ???? X
  plt.plot(fpr, tpr,
            color=clr,
            linestyle=ls,
            label='%s (auc = %0.3f)' % (label, roc_auc))
  
plt.legend(loc='lower right')
plt.plot([0, 1], [0, 1],
          linestyle='--',
          color='gray',
          linewidth=2)

plt.xlim([-0.1, 1.1])
plt.ylim([-0.1, 1.1])
plt.grid(alpha=0.5)
plt.xlabel('False positive rate (FPR)')
plt.ylabel('True positive rate (TPR)')

plt.show()

### ?? ???? ??

In [ ]:
voting.fit(X_train, y_train)
y_pred = voting.predict(X_test)

print('?? ??? ?? ??: %d' % (y_test != y_pred).sum())
print('???: %.3f' % accuracy_score(y_test, y_pred))
print('???: %.3f' % precision_score(y_true=y_test, y_pred=y_pred))
print('???: %.3f' % recall_score(y_true=y_test, y_pred=y_pred))
print('F1: %.3f' % f1_score(y_true=y_test, y_pred=y_pred))

# ??

## adaboost

In [ ]:
adaboost.get_params()

In [ ]:
parms = {'n_estimators': [100,200,300]
             }


grid = GridSearchCV(estimator=adaboost, 
                    param_grid=parms,
                    cv=10,
                    scoring='roc_auc',
                    iid=False)

grid.fit(X_train,y_train)

for r, _ in enumerate(grid.cv_results_['mean_test_score']):
  print("%0.3f +/- %0.3f %r" % (grid.cv_results_['mean_test_score'][r],
        grid.cv_results_['std_test_score'][r] / 2.0,
        grid.cv_results_['params'][r]))

In [ ]:
print('??? ????: %s' % grid.best_params_)
print('ACU: %.3f' % grid.best_score_)

## voting

In [ ]:
parms = {'logistic__C':[0.001,0.1,100.0],
         'tree__max_depth' :  [1,2,3,4,5],
         'knn__n_neighbors':[1,2,3,4,5]}


grid = GridSearchCV(estimator=voting, 
                    param_grid=parms,
                    cv=10,
                    scoring='roc_auc',
                    iid=False)


In [ ]:
grid.fit(X_train,y_train)

In [ ]:
for r, _ in enumerate(grid.cv_results_['mean_test_score']):
  print("%0.3f +/- %0.3f %r" % (grid.cv_results_['mean_test_score'][r],
        grid.cv_results_['std_test_score'][r] / 2.0,
        grid.cv_results_['params'][r]))


In [ ]:
print('??? ????: %s' % grid.best_params_)
print('ACU: %.3f' % grid.best_score_)
